[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/minimind-colab/blob/main/notebooks/07_Inference_and_Deployment.ipynb)

# Module 7: Inference & Deployment

In [ ]:
# @title 🔧 Environment Setup (Run this first!)
import os

gpu_info = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU"
print(f"GPU: {gpu_info[0]}")

if not os.path.exists('minimind-colab'):
    !git clone https://github.com/Boyu-Zhang-UOI/minimind-colab.git
os.chdir('minimind-colab')

!pip install -q transformers==4.57.6 datasets==3.6.0 jinja2==3.1.2 tokenizers fastapi uvicorn

import sys
sys.path.insert(0, '.')
print("✅ Setup complete!")

## 📚 Overview

In the final module, we bring everything together — from trained model to deployed application:

1. **Autoregressive generation** — How `model.generate()` works under the hood
2. **Sampling strategies** — Temperature, top-k, top-p, repetition penalty
3. **KV-Cache** — Efficient inference by reusing past computations
4. **Interactive CLI chat** — Multi-turn conversations with streaming
5. **OpenAI-compatible API** — Deploy as a REST API server
6. **Web UI** — Streamlit-based chat interface
7. **Model conversion** — Export to HuggingFace format
8. **Course wrap-up** — Review the complete pipeline

**Key source files:** `model/model_minimind.py` (generate method), `eval_llm.py`, `scripts/serve_openai_api.py`, `scripts/web_demo.py`, `scripts/convert_model.py`

## 1. Autoregressive Generation

The `generate()` method in `MiniMindForCausalLM` produces text one token at a time:

```
Input: "The sky is"
  Step 1: Predict next token → "blue" (append)
  Step 2: Predict next token → "because" (append)
  Step 3: Predict next token → "of" (append)
  ...until EOS or max_new_tokens reached
```

At each step:
1. Forward pass → logits for all vocabulary tokens
2. Apply temperature scaling
3. Apply top-k / top-p filtering
4. Apply repetition penalty
5. Sample from filtered distribution
6. Check for EOS token

**KV-Cache**: Instead of reprocessing all tokens at each step, we cache the Key and Value matrices from previous tokens. Each new step only computes Q, K, V for the **single new token** and concatenates with the cache.

In [ ]:
import torch
import sys
sys.path.insert(0, '.')
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from transformers import AutoTokenizer

config = MiniMindConfig(hidden_size=768, num_hidden_layers=8)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = MiniMindForCausalLM(config).to(device).half().eval()
tokenizer = AutoTokenizer.from_pretrained('./model')

print(f"Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params on {device}")

## 2. Sampling Strategies

| Strategy | Effect | Parameter |
|----------|--------|-----------|
| **Temperature** | Scale logits by `1/T` before softmax | `temperature=0.85` |
| **Top-k** | Keep only top k tokens by probability | `top_k=50` |
| **Top-p** | Keep tokens until cumulative prob ≥ p | `top_p=0.95` |
| **Repetition penalty** | Divide logits of already-seen tokens | `repetition_penalty=1.0` |

Lower temperature → more deterministic (confident).
Higher temperature → more creative (random).

In [ ]:
# Demonstrate sampling parameter effects
prompt = "The meaning of life is"
inputs = tokenizer(tokenizer.bos_token + prompt, return_tensors="pt").to(device)

configs = [
    {"temperature": 0.1, "top_p": 0.95, "label": "T=0.1 (deterministic)"},
    {"temperature": 0.85, "top_p": 0.95, "label": "T=0.85 (balanced)"},
    {"temperature": 1.5, "top_p": 0.99, "label": "T=1.5 (creative)"},
]

print(f"Prompt: '{prompt}'\n")
for cfg in configs:
    with torch.no_grad():
        output_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=40,
            temperature=cfg["temperature"],
            top_p=cfg["top_p"],
            do_sample=True,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print(f"[{cfg['label']}]")
    print(f"  {text}\n")

## 3. Interactive CLI Chat

The `eval_llm.py` script provides an interactive chat interface:

```python
# Key components:
conversation = []
for prompt in user_inputs:
    conversation.append({"role": "user", "content": prompt})
    
    # Apply chat template
    formatted = tokenizer.apply_chat_template(
        conversation, tokenize=False, add_generation_prompt=True
    )
    
    # Generate with streaming
    output = model.generate(inputs, streamer=streamer, ...)
    
    conversation.append({"role": "assistant", "content": response})
```

Features:
- Multi-turn conversation history
- Streaming output (token-by-token display)
- Speed measurement (tokens/second)
- LoRA adapter loading at inference time

In [ ]:
from transformers import TextStreamer
import time

def chat(model, tokenizer, prompt, history=None, max_new_tokens=100):
    """Simple chat function with streaming."""
    if history is None:
        history = []
    
    history.append({"role": "user", "content": prompt})
    
    formatted = tokenizer.apply_chat_template(
        history, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(formatted, return_tensors="pt", truncation=True).to(device)
    
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    start = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.85,
            top_p=0.95,
            do_sample=True,
            streamer=streamer,
        )
    elapsed = time.time() - start
    
    response = tokenizer.decode(
        output_ids[0][inputs.input_ids.shape[1]:], 
        skip_special_tokens=True
    )
    gen_tokens = output_ids.shape[1] - inputs.input_ids.shape[1]
    print(f"\n[{gen_tokens} tokens in {elapsed:.2f}s = {gen_tokens/elapsed:.1f} tok/s]")
    
    history.append({"role": "assistant", "content": response})
    return history

# Demo conversation
print("=== Interactive Chat Demo ===\n")
history = None
for prompt in ["What is AI?", "Tell me more about deep learning."]:
    print(f"💬 User: {prompt}")
    print("🧠 Assistant: ", end="")
    history = chat(model, tokenizer, prompt, history, max_new_tokens=50)
    print()

## 4. OpenAI-Compatible API Server

`scripts/serve_openai_api.py` creates a FastAPI server that matches the OpenAI API format:

```
POST /v1/chat/completions
{
    "model": "minimind",
    "messages": [{"role": "user", "content": "Hello"}],
    "temperature": 0.7,
    "stream": true
}
```

Key features:
- **Streaming**: Server-Sent Events (SSE) for token-by-token output
- **Thinking**: Extracts `<think>...</think>` as `reasoning_content`
- **Tool calls**: Parses `<tool_call>...</tool_call>` into structured format
- **OpenAI compatibility**: Works with the official `openai` Python client

In [ ]:
# Show how the API response parsing works
import re
import json

def parse_response(text):
    """Parse model output into content, reasoning, and tool calls."""
    reasoning_content = None
    
    # Extract thinking content
    think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
    if think_match:
        reasoning_content = think_match.group(1).strip()
        text = re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)
    
    # Extract tool calls
    tool_calls = []
    for i, m in enumerate(re.findall(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL)):
        try:
            call = json.loads(m.strip())
            tool_calls.append({
                "id": f"call_{i}",
                "type": "function",
                "function": {
                    "name": call.get("name", ""),
                    "arguments": json.dumps(call.get("arguments", {}))
                }
            })
        except (json.JSONDecodeError, ValueError):
            pass
    
    if tool_calls:
        text = re.sub(r'<tool_call>.*?</tool_call>', '', text, flags=re.DOTALL)
    
    return text.strip(), reasoning_content, tool_calls or None

# Demo parsing
test_outputs = [
    "The answer is 42.",
    '<think>\nLet me think step by step.\n1. First, consider...\n</think>\nThe answer is 42.',
    '<tool_call>\n{"name": "calculate_math", "arguments": {"expression": "2+2"}}\n</tool_call>',
]

print("=== Response Parsing Demo ===\n")
for output in test_outputs:
    content, reasoning, tools = parse_response(output)
    print(f"Input: {output[:60]}...")
    print(f"  Content: {content}")
    print(f"  Reasoning: {reasoning}")
    print(f"  Tool calls: {tools}")
    print()

## 5. Web UI & Model Conversion

### Streamlit Web Demo
`scripts/web_demo.py` provides a full chat UI with:
- Multi-turn conversation
- Thinking display (expand/collapse)
- Tool call visualization
- Model/parameter settings
- Real-time streaming

### Model Conversion
`scripts/convert_model.py` converts PyTorch `.pth` checkpoints to HuggingFace format:
- Enables uploading to HuggingFace Hub
- Enables using with the `transformers` ecosystem
- Enables easy sharing and collaboration

In [ ]:
# Show model conversion structure
print("=== Model Conversion: PyTorch → HuggingFace ===\n")

# What convert_model.py does:
print("1. Load PyTorch checkpoint (.pth)")
print("   state_dict = torch.load('out/full_sft_768.pth')")
print()
print("2. Create model with config")
print("   model = MiniMindForCausalLM(config)")
print("   model.load_state_dict(state_dict)")
print()
print("3. Save in HuggingFace format")
print("   model.save_pretrained('minimind-hf/')")
print("   tokenizer.save_pretrained('minimind-hf/')")
print()
print("4. Result: HuggingFace-compatible directory")
print("   minimind-hf/")
print("   ├── config.json")
print("   ├── model.safetensors")
print("   ├── tokenizer.json")
print("   └── tokenizer_config.json")
print()

# Show the actual model config that would be saved
print("=== Model Config ===")
config_dict = config.to_dict()
for key in ['hidden_size', 'num_hidden_layers', 'num_attention_heads', 
            'num_key_value_heads', 'vocab_size', 'intermediate_size',
            'max_position_embeddings', 'hidden_act']:
    print(f"  {key}: {config_dict.get(key)}")

## 6. Course Wrap-Up — The Complete Pipeline

Over 7 modules, we built an LLM from scratch:

| Module | What We Built | Key Concept |
|--------|--------------|-------------|
| **1. Tokenizer** | BPE vocabulary (6400 tokens) | Text → Numbers |
| **2. Architecture** | RMSNorm, RoPE, GQA, MoE | Transformer building blocks |
| **3. Pretraining** | Causal LM on raw text | Next-token prediction |
| **4. SFT + LoRA** | Instruction following | Label masking, low-rank adaptation |
| **5. Alignment** | DPO + GRPO | Preference learning, policy optimization |
| **6. Advanced** | Agent RL + Distillation | Tool use, knowledge transfer |
| **7. Deployment** | CLI, API, Web UI | From model to application |

```
Raw Text → Tokenizer → Pretrain → SFT → Align → Deploy
              ↓           ↓         ↓       ↓       ↓
           6400 vocab   64M LM   Chat   Prefer   API/UI
```

## ✏️ Exercises

1. **Sampling exploration**: Generate 5 responses with `temperature=0.1` and 5 with `temperature=1.5` for the same prompt. Which set is more diverse? Which is more coherent?

2. **Speed benchmark**: Measure tokens/second for different sequence lengths (50, 100, 200, 500 tokens). How does KV-Cache affect generation speed?

3. **API client**: If running the API server, use the `openai` Python client to send requests. Try both streaming and non-streaming modes.

4. **End-to-end project**: Starting fresh, run the complete pipeline from tokenizer training through deployment. Document your experience.

## 🎓 Final Take-Home Projects (Optional)

1. **Custom domain adaptation**: Pick a domain (code, medical, legal). Collect a small dataset, run LoRA fine-tuning, and evaluate.

2. **Extend the tool set**: Add a new tool to `train_agent.py` (e.g., Wikipedia search, code executor). Train the model to use it.

3. **Scale up**: Try `hidden_size=1024` or `num_hidden_layers=16`. How does the quality change with more parameters?

4. **Write a blog post**: Explain one training stage (pretraining, SFT, DPO, or GRPO) in your own words with diagrams.

## 📝 Summary

In this final module we covered:

- **Autoregressive generation** — Token-by-token prediction with KV-Cache
- **Sampling strategies** — Temperature, top-k, top-p, repetition penalty
- **Interactive chat** — Multi-turn conversations with streaming output
- **OpenAI API** — REST endpoint matching the OpenAI format
- **Response parsing** — Extracting thinking, content, and tool calls
- **Web UI** — Streamlit interface for visual interaction
- **Model conversion** — PyTorch → HuggingFace format for sharing
- **Complete pipeline** — Tokenizer → Pretrain → SFT → Align → Deploy

🎉 **Congratulations!** You have built a complete LLM from scratch — from raw text to deployed application!